# FERRUS OSSEUS — Fregate 04
## FLOTTE FERRUS | AD MAJOREM GLORIAM IMPERATORIS

**Pipeline** : Mesh T-pose (sans rig) → FBX rigé (R15 | Mixamo | DeepMotion)

**Tout vit sur Google Drive** — le repo, les inputs, les outputs.
Entre chaque session Colab, rien n'est perdu.

```
MyDrive/FLOTTE-FERRUS/04_FERRUS_OSSEUS/IN/   <- deposer l'avatar ici
MyDrive/FLOTTE-FERRUS/04_FERRUS_OSSEUS/OUT/  <- recuperer le FBX rige ici
```

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLULE 1 — CONFIGURATION
# Seule cellule a editer. Tout le reste est automatique.
# ══════════════════════════════════════════════════════════════════════════

# ── Fichier source ────────────────────────────────────────────────────────
INPUT_FILE = "avatar_P1.glb"    # Nom du fichier que tu as depose dans Drive/IN/
                                 # Formats : .glb  .gltf  .obj  .fbx

# ── Template squelette ───────────────────────────────────────────────────
TEMPLATE = "r15"                 # r15 | mixamo | deepmotion

# ── Acces GitHub (repo prive) ────────────────────────────────────────────
GITHUB_TOKEN = "TON_TOKEN_ICI"
GITHUB_REPO  = "kioka8877-ux/FLOTTE-FERRUS"

# ══════════════════════════════════════════════════════════════════════════
# NE PAS MODIFIER EN DESSOUS
# ══════════════════════════════════════════════════════════════════════════
DRIVE_REPO  = "/content/drive/MyDrive/FLOTTE-FERRUS"          # repo sur Drive
FREGATE_DIR = f"{DRIVE_REPO}/04_FERRUS_OSSEUS"                # fregate
IN_DIR      = f"{FREGATE_DIR}/IN"                             # ← deposer ici
OUT_DIR     = f"{FREGATE_DIR}/OUT"                            # ← recuperer ici
SCRIPT_PATH = f"{FREGATE_DIR}/codebase/osseus_core.py"        # script

print("[OSSEUS] Configuration OK")
print(f"  Template : {TEMPLATE}")
print(f"  Input    : {INPUT_FILE}")
print(f"  Drive IN : {IN_DIR}")
print(f"  Drive OUT: {OUT_DIR}")
print()
print("[OSSEUS] Executer C2 pour initialiser Drive et cloner le repo.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLULE 2 — SETUP DRIVE + REPO
# Monte Drive, clone le repo dessus (ou git pull si deja present).
# Apres cette cellule : Drive/FLOTTE-FERRUS/ existe avec tout le code.
# ══════════════════════════════════════════════════════════════════════════

from google.colab import drive
import subprocess, os

def run(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    out = (r.stdout + r.stderr).strip()
    if out:
        # Filtrer les lignes Blender/Git verboses, garder l'essentiel
        for line in out.split("\n")[-10:]:
            if line.strip():
                print(f"  {line}")
    return r

# ── 1. Monter Google Drive ────────────────────────────────────────────────
print("[OSSEUS] Montage Google Drive...")
drive.mount("/content/drive")
print("[OSSEUS] Drive monte.")

# ── 2. Cloner ou mettre a jour le repo SUR Drive ─────────────────────────
git_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"

if not os.path.exists(f"{DRIVE_REPO}/.git"):
    print(f"\n[OSSEUS] Premiere initialisation — clone du repo sur Drive...")
    print(f"  Destination : {DRIVE_REPO}")
    run(f"git clone {git_url} '{DRIVE_REPO}'")
    print("[OSSEUS] Clone termine.")
else:
    print(f"\n[OSSEUS] Repo present sur Drive — git pull (mise a jour du code)...")
    # Mettre a jour l'URL au cas ou le token a change
    run(f"git remote set-url origin {git_url}", cwd=DRIVE_REPO)
    run("git pull --rebase", cwd=DRIVE_REPO)
    print("[OSSEUS] Code a jour.")

# ── 3. Creer IN/ et OUT/ si absents ──────────────────────────────────────
for d in [IN_DIR, OUT_DIR, f"{FREGATE_DIR}/codebase/docs"]:
    os.makedirs(d, exist_ok=True)

# ── 4. Verifier que le script est bien present ────────────────────────────
if not os.path.exists(SCRIPT_PATH):
    raise RuntimeError(f"Script manquant apres clone : {SCRIPT_PATH}")

# ── 5. Bilan ─────────────────────────────────────────────────────────────
in_files  = [f for f in os.listdir(IN_DIR)  if not f.startswith('.')]
out_files = [f for f in os.listdir(OUT_DIR) if not f.startswith('.')]

print(f"\n[OSSEUS] Drive pret.")
print(f"  Repo    : {DRIVE_REPO}")
print(f"  Script  : {SCRIPT_PATH}")
print(f"  IN/     : {in_files if in_files else '(vide — deposer un avatar)'}")
print(f"  OUT/    : {out_files if out_files else '(vide)'}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLULE 3 — INSTALLATION BLENDER
# A relancer une seule fois par session Colab (pas besoin si deja installe).
# ══════════════════════════════════════════════════════════════════════════

import subprocess

check = subprocess.run("blender --version", shell=True, capture_output=True, text=True)

if check.returncode == 0:
    ver = check.stdout.strip().split("\n")[0]
    print(f"[OSSEUS] Blender deja installe : {ver}")
else:
    print("[OSSEUS] Installation Blender (~2 min, une seule fois par session)...")
    r = subprocess.run("apt-get install -y blender 2>&1 | tail -5",
                       shell=True, capture_output=True, text=True)
    print(r.stdout[-500:] if r.stdout else r.stderr[-500:])

    ver_check = subprocess.run("blender --version", shell=True, capture_output=True, text=True)
    if ver_check.returncode != 0:
        raise RuntimeError("Blender non installe. Verifier les droits sudo de la session.")
    print(f"[OSSEUS] Blender installe : {ver_check.stdout.strip().split(chr(10))[0]}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLULE 4 — VERIFICATION INPUT
# Verifie que INPUT_FILE est bien dans Drive/IN/.
# Propose un upload interactif si absent.
# ══════════════════════════════════════════════════════════════════════════

import os
from pathlib import Path

input_full_path = os.path.join(IN_DIR, INPUT_FILE)

if os.path.exists(input_full_path):
    size_mb = os.path.getsize(input_full_path) / (1024*1024)
    print(f"[OSSEUS] Input trouve sur Drive : {INPUT_FILE} ({size_mb:.2f} Mo)")
    print(f"[OSSEUS] Pret pour le pipeline.")
else:
    print(f"[OSSEUS] '{INPUT_FILE}' introuvable dans Drive/IN/")
    print(f"[OSSEUS] IN/ = {IN_DIR}")
    print()
    print("Options :")
    print("  A) Deposer le fichier directement dans Drive depuis ton PC")
    print("     puis relancer cette cellule")
    print("  B) Upload via Colab ci-dessous")
    print()

    from google.colab import files
    print("[OSSEUS] Upload du fichier...")
    uploaded = files.upload()

    for fname, data in uploaded.items():
        dest = os.path.join(IN_DIR, fname)
        with open(dest, 'wb') as f:
            f.write(data)
        size_mb = len(data) / (1024*1024)
        print(f"[OSSEUS] Sauvegarde sur Drive : IN/{fname} ({size_mb:.2f} Mo)")
        if len(uploaded) == 1:
            INPUT_FILE      = fname
            input_full_path = dest

    if not os.path.exists(input_full_path):
        raise RuntimeError(f"Aucun fichier uploade correspondant a '{INPUT_FILE}'")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLULE 5 — PIPELINE OSSEUS
# Rigging automatique : detection bbox → placement squelette → auto-weights
# La sortie FBX est sauvegardee directement dans Drive/OUT/
# ══════════════════════════════════════════════════════════════════════════

import os, subprocess, json
from pathlib import Path

stem        = Path(INPUT_FILE).stem
output_file = f"avatar_rigged_{stem}_{TEMPLATE}.fbx"
output_path = os.path.join(OUT_DIR, output_file)
report_path = os.path.join(OUT_DIR, f"rapport_osseus_{stem}.json")

print(f"[OSSEUS] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"[OSSEUS] DEBUT PIPELINE")
print(f"[OSSEUS]   Input    : {INPUT_FILE}")
print(f"[OSSEUS]   Template : {TEMPLATE}")
print(f"[OSSEUS]   Sortie   : Drive/OUT/{output_file}")
print(f"[OSSEUS] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

cmd = (
    f"blender --background --python '{SCRIPT_PATH}' -- "
    f"--input   '{input_full_path}' "
    f"--output  '{output_path}' "
    f"--template {TEMPLATE} "
    f"--report  '{report_path}'"
)

result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

# Afficher uniquement les logs OSSEUS
all_output  = result.stdout + result.stderr
osseus_logs = [l for l in all_output.split("\n") if l.startswith("[OSSEUS]")]
print("\n".join(osseus_logs))

if result.returncode != 0 and not os.path.exists(output_path):
    print("\n[OSSEUS] ERREUR — Logs Blender complets :")
    print(all_output[-3000:])
    raise RuntimeError("Pipeline OSSEUS echoue")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLULE 6 — RAPPORT ET VALIDATION
# ══════════════════════════════════════════════════════════════════════════

import json, os

if os.path.exists(report_path):
    with open(report_path) as f:
        rapport = json.load(f)

    status = rapport.get('status', 'UNKNOWN')
    ok     = status == 'SUCCESS'

    print(f"[OSSEUS] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"[OSSEUS] RAPPORT — {'OK' if ok else 'ECHEC'} ({status})")
    print(f"[OSSEUS] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"  Input         : {rapport.get('input', '-')}")
    print(f"  Template      : {rapport.get('template', '-')}")
    print(f"  Vertices      : {rapport.get('vertices', 0):,}")
    print(f"  Faces         : {rapport.get('faces', 0):,}")
    print(f"  Hauteur mesh  : {rapport.get('height', '-')}")
    print(f"  Largeur mesh  : {rapport.get('width', '-')}")
    print(f"  Bones crees   : {rapport.get('bones_count', '-')}")
    print(f"  Auto Weights  : {rapport.get('auto_weights', '-')}")
    print(f"  Taille sortie : {rapport.get('output_size_mb', '-')} Mo")

    if not ok:
        print(f"\n  ERREUR : {rapport.get('error', 'inconnue')}")
else:
    print("[OSSEUS] Rapport JSON introuvable")

if os.path.exists(output_path):
    size = os.path.getsize(output_path) / (1024*1024)
    print(f"\n[OSSEUS] FBX rige sur Drive : OUT/{output_file} ({size:.2f} Mo)")
    print(f"[OSSEUS] Prochaine etape : copier dans FERRUS ANIMUS IN/")
else:
    print("\n[OSSEUS] ERREUR : FBX non genere")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLULE 7 — TRANSFERT VERS FERRUS ANIMUS (optionnel)
# Copie le FBX rige directement dans Drive/01_FERRUS_ANIMUS/IN/
# ══════════════════════════════════════════════════════════════════════════

import os, shutil

TRANSFERT_VERS_ANIMUS = False   # Passer a True pour copie auto

ANIMUS_IN_DIR = f"{DRIVE_REPO}/01_FERRUS_ANIMUS/IN"

if TRANSFERT_VERS_ANIMUS and os.path.exists(output_path):
    os.makedirs(ANIMUS_IN_DIR, exist_ok=True)
    dest = os.path.join(ANIMUS_IN_DIR, output_file)
    shutil.copy2(output_path, dest)
    print(f"[OSSEUS] Transfere vers FERRUS ANIMUS :")
    print(f"  {dest}")
    print(f"[OSSEUS] Ouvrir 01_FERRUS_ANIMUS pour continuer.")
else:
    if not TRANSFERT_VERS_ANIMUS:
        print("[OSSEUS] Transfert desactive. Mettre TRANSFERT_VERS_ANIMUS=True pour activer.")
    print(f"[OSSEUS] FBX disponible sur Drive : OUT/{output_file}")